<a href="https://colab.research.google.com/github/YokoyamaLab/PythonBasics/blob/2025/27_day08tb_YoloDetect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 必要なライブラリのインストール
!pip install ultralytics
!pip install ipywidgets

# ライブラリのインポート
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display, Image
import os

In [ ]:
# YOLOv11モデルのロード
# 'yolov11x.pt' は特大モデル、他のモデル（s, m, l, x）も選択可能
model = YOLO('yolo11x.pt')

# ファイルアップロードウィジェットの作成
uploader = widgets.FileUpload(
    accept='.jpg,.jpeg',  # 受け付けるファイルの種類
    multiple=False  # 単一ファイルのみ
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        output.clear_output()
        if uploader.value:
            uploaded_file_name = list(uploader.value.keys())[0]
            uploaded_file_content = uploader.value[uploaded_file_name]['content']

            # アップロードされたファイルを一時的に保存
            with open(uploaded_file_name, 'wb') as f:
                f.write(uploaded_file_content)

            print(f"ファイルをアップロードしました: {uploaded_file_name}")

            try:
                # YOLOによる物体認識の実行
                # save=Trueで結果画像をruns/detect/exp*/ に保存
                results = model(uploaded_file_name, save=True)

                # 結果が保存されたパスを取得
                # 通常、runs/detect/expN/uploaded_file_name の形式で保存されます
                # runs/detect ディレクトリ内の最新のディレクトリを取得
                runs_dir = 'runs/detect'
                if os.path.exists(runs_dir):
                    exp_dirs = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir) if os.path.isdir(os.path.join(runs_dir, d))]
                    exp_dirs.sort(key=os.path.getmtime, reverse=True) # 最新のディレクトリを先頭に
                    if exp_dirs:
                        result_image_path = os.path.join(exp_dirs[0], uploaded_file_name)

                        # 結果画像の表示
                        print("物体認識結果:")
                        display(Image(filename=result_image_path))

                        # 生成された一時ファイルを削除 (オプション)
                        # os.remove(uploaded_file_name)
                        # os.remove(result_image_path) # 結果画像を削除したい場合
                    else:
                        print("結果ディレクトリが見つかりませんでした。")
                else:
                    print("runs/detect ディレクトリが見つかりませんでした。")

            except Exception as e:
                print(f"物体認識中にエラーが発生しました: {e}")
                if os.path.exists(uploaded_file_name):
                     os.remove(uploaded_file_name)


# ファイルアップロードウィジェットの変更を監視
uploader.observe(on_upload_change, names='value')

# ウィジェットの表示
print("JPGファイルをアップロードしてください:")
display(uploader, output)
